# RSVP / TARTAN / CLIO Field Laboratory

Unified experimental notebook.  Each section is self-contained.
Run cells top-to-bottom; shared fields are reset per experiment.

**Modules used**
```
operators/rsvp.py       — fields, evolution, fractional Laplacian
operators/clio.py       — repair variants (standard/sheaf/identity/economic)
operators/semantic.py   — attractor basins, observer projections
operators/topology.py   — sheaf metrics, TARTAN, PCA embedding
operators/cosmology.py  — Hubble expansion, coupled manifolds, economy
```

## Cell 1 — Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from operators.rsvp       import (init_fields, evolve, evolve_fractional,
                                   admissibility, field_metrics,
                                   coarse_grain, inject_kink,
                                   gradient, curl, laplacian)
from operators.clio       import (clio_standard, clio_sheaf,
                                   clio_identity, get_clio)
from operators.semantic   import (SemanticField, observer_projection,
                                   anisotropy_index)
from operators.topology   import (gluing_obstruction, local_gluing_map,
                                   connected_components, euler_characteristic,
                                   tartan_tiles, tartan_hierarchy,
                                   pca_embedding, synchrony, gini,
                                   structure_function)
from operators.cosmology  import (evolve_cosmological, CosmologicalHistory,
                                   evolve_coupled, EconomicField,
                                   double_well_force)

plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 11})
H, W = 128, 128
DT   = 0.05
print('Imports OK')


## Cell 2 — Baseline RSVP Simulation

1000-step run with standard CLIO repair.  Fields Φ, S, and torsion ω
are saved every 10 steps for downstream analysis.

In [ ]:
np.random.seed(42)
phi, vx, vy, S, R = init_fields(H, W, seed=42)

clio_fn = get_clio('standard', lambda_repair=0.15, smooth_sigma=1.2)

history_phi     = []
history_entropy = []
history_torsion = []
metrics_log     = []

for step in range(1000):
    phi, vx, vy, S, R = evolve(phi, vx, vy, S, R, dt=DT, clio_fn=clio_fn)
    if step % 10 == 0:
        history_phi.append(phi.copy())
        history_entropy.append(S.copy())
        history_torsion.append(curl(vx, vy))
        metrics_log.append(field_metrics(phi, S, vx, vy))

print(f'Saved {len(history_phi)} frames')
print('Final metrics:', {k: f"{v:.4f}" for k, v in metrics_log[-1].items()})


## Cell 3 — Animated Field Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
ax1, ax2, ax3 = axes

im1 = ax1.imshow(history_phi[0],     cmap='inferno',  animated=True)
im2 = ax2.imshow(history_entropy[0], cmap='viridis',  animated=True)
im3 = ax3.imshow(history_torsion[0], cmap='coolwarm', animated=True)

ax1.set_title('Scalar Field Φ')
ax2.set_title('Entropy Field S')
ax3.set_title('Torsion / Memory ω')
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])

def update(frame):
    im1.set_array(history_phi[frame])
    im2.set_array(history_entropy[frame])
    im3.set_array(history_torsion[frame])
    return im1, im2, im3

anim = FuncAnimation(fig, update, frames=len(history_phi), interval=50, blit=False)
plt.tight_layout()
HTML(anim.to_jshtml())


## Cell 4 — Phase Metrics & Admissibility

In [ ]:
phi_energy     = [m['phi_energy']    for m in metrics_log]
entropy_energy = [m['entropy_mean']  for m in metrics_log]
torsion_energy = [m['torsion_mean']  for m in metrics_log]
adm_frac       = [m['admissible_frac'] for m in metrics_log]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(phi_energy,     label='Φ energy')
ax1.plot(entropy_energy, label='Entropy S')
ax1.plot(torsion_energy, label='Torsion |ω|')
ax1.legend(); ax1.set_title('RSVP Dynamical Metrics')
ax1.set_xlabel('Frame'); ax1.set_ylabel('Magnitude')

ax2.plot(adm_frac, color='seagreen')
ax2.set_title('Admissible Fraction over Time')
ax2.set_xlabel('Frame'); ax2.set_ylabel('Fraction')
ax2.set_ylim(0, 1)

plt.tight_layout(); plt.show()

adm_map = admissibility(phi, S)
plt.figure(figsize=(6, 6))
plt.imshow(adm_map, cmap='gray')
plt.title('Admissible Trajectory Regions (final)')
plt.axis('off'); plt.tight_layout(); plt.show()


## Cell 5 — Sheaf Gluing Obstruction

Track the global cocycle error Ω(t) = mean ||φᵢ|ᵢⱼ − φⱼ|ᵢⱼ||²
and visualise the spatial obstruction map at the final frame.

In [ ]:
obstruction_curve = [gluing_obstruction(f, patch=32, overlap=8)
                     for f in history_phi]

plt.figure(figsize=(10, 4))
plt.plot(obstruction_curve, color='crimson')
plt.axhline(np.mean(obstruction_curve), ls='--', color='gray', label='mean')
plt.title('Sheaf Gluing Obstruction Ω(t)')
plt.xlabel('Frame'); plt.ylabel('||cocycle||²')
plt.legend(); plt.tight_layout(); plt.show()

obs_map = local_gluing_map(history_phi[-1], patch=32, overlap=8)
plt.figure(figsize=(7, 6))
plt.imshow(obs_map, cmap='hot', interpolation='nearest')
plt.colorbar(label='Local Gluing Error')
plt.title('Spatial Obstruction Map (final frame)')
plt.axis('off'); plt.tight_layout(); plt.show()


## Cell 6 — TARTAN Recursive Tiling

Multi-scale tile statistics at 3 levels of resolution.

In [ ]:
hier = tartan_hierarchy(history_phi[-1], levels=3, base_tile=8)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for lvl in range(3):
    tiles     = hier[lvl]
    tile_size = 8 * (2 ** lvl)
    means     = [t['mean']     for t in tiles]
    variances = [t['variance'] for t in tiles]
    skews     = [t['skew']     for t in tiles]
    axes[0, lvl].hist(means,     bins=20, color='steelblue')
    axes[0, lvl].set_title(f'Level {lvl} — tile {tile_size}px — Means')
    axes[1, lvl].hist(variances, bins=20, color='tomato')
    axes[1, lvl].set_title(f'Level {lvl} — Variances')

plt.suptitle('TARTAN Recursive Tile Statistics', fontsize=13)
plt.tight_layout(); plt.show()

# Structure function — scale-invariance check
lags, D = structure_function(history_phi[-1], max_lag=40)
plt.figure(figsize=(8, 4))
plt.loglog(lags, D, marker='o', ms=4)
plt.title('Structure Function D(r) = ⟨(Φ(x+r)−Φ(x))²⟩')
plt.xlabel('Lag r (pixels)'); plt.ylabel('D(r)')
plt.tight_layout(); plt.show()


## Cell 7 — Level-Set Topology

Connected components, Euler characteristic, and a crude
persistence diagram across threshold levels.

In [ ]:
thresholds = np.linspace(history_phi[-1].min(), history_phi[-1].max(), 30)
n_comps  = [connected_components(history_phi[-1], level=t) for t in thresholds]
eulers   = [euler_characteristic(history_phi[-1], level=t) for t in thresholds]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(thresholds, n_comps, color='steelblue')
ax1.set_title('Connected Components vs Threshold')
ax1.set_xlabel('Level'); ax1.set_ylabel('# components')

ax2.plot(thresholds, eulers, color='darkorange')
ax2.axhline(0, ls='--', color='gray')
ax2.set_title('Euler Characteristic vs Threshold')
ax2.set_xlabel('Level'); ax2.set_ylabel('χ')
plt.tight_layout(); plt.show()

pairs = [p for p in __import__('operators.topology', fromlist=['persistence_diagram'])
         .__class__.__mro__]  # placeholder — call directly:
from operators.topology import persistence_diagram
pairs = persistence_diagram(history_phi[-1], n_levels=30)
if pairs:
    births, deaths = zip(*pairs)
    plt.figure(figsize=(6, 6))
    plt.scatter(births, deaths, alpha=0.6, s=20)
    lo = min(births); hi = max(deaths)
    plt.plot([lo, hi], [lo, hi], 'k--', lw=0.8)
    plt.title('0-D Persistence Diagram')
    plt.xlabel('Birth'); plt.ylabel('Death')
    plt.tight_layout(); plt.show()


## Cell 8 — Semantic Attractor Fields

Eight named concept-tokens modelled as Coulomb charges.
Track basin assignment, inter-basin flux, and per-basin
scalar-field commitment (variance within basin).

In [ ]:
LABELS = ['RSVP', 'entropy', 'constraint', 'soliton',
          'closure', 'repair', 'torsion', 'trajectory']

sem = SemanticField(H, W, n_tokens=8, labels=LABELS, seed=3)

phi_s, vx_s, vy_s, S_s, R_s = init_fields(H, W, seed=5)
# Seed the field with the attractor landscape
phi_s += 0.3 * sem.potential()

from operators.semantic import evolve_semantic
sem_frames = []
flux_history = []

clio_sem = get_clio('standard', lambda_repair=0.12)
for step in range(500):
    phi_s, vx_s, vy_s, S_s, R_s = evolve_semantic(
        phi_s, vx_s, vy_s, S_s, R_s,
        sem_field=sem, dt=DT, gamma_sem=0.25, clio_fn=clio_sem
    )
    if step % 25 == 0:
        sem_frames.append(phi_s.copy())
        flux_history.append(sem.inter_basin_flux(phi_s))

# Final landscape
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.imshow(sem_frames[-1], cmap='twilight', origin='upper')
for (px, py), q, lbl in zip(sem.positions, sem.charges, sem.labels):
    col = 'white' if q > 0 else 'yellow'
    ax1.scatter(px, py, c=col, s=100, zorder=5, edgecolors='black')
    ax1.text(px+2, py-4, lbl, color=col, fontsize=8,
             bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.5))
ax1.set_title('Semantic Attractor Landscape (final Φ)')
ax1.axis('off')

bmap = sem.basin_map()
ax2.imshow(bmap, cmap='tab10', alpha=0.7, origin='upper')
ax2.imshow(sem_frames[-1], cmap='gray', alpha=0.25, origin='upper')
for (px, py), lbl in zip(sem.positions, sem.labels):
    ax2.text(px, py, lbl, fontsize=7, ha='center', va='center',
             color='white', fontweight='bold')
ax2.set_title('Basin Partition')
ax2.axis('off')
plt.tight_layout(); plt.show()

plt.figure(figsize=(9, 3))
plt.plot(flux_history, color='mediumvioletred')
plt.title('Inter-Basin Flux |∇Φ| at Boundaries')
plt.xlabel('Frame'); plt.ylabel('Mean |∇Φ|')
plt.tight_layout(); plt.show()


## Cell 9 — Observer-Dependent Projections

Four observers project Φ and S along different lines of sight.
Anisotropy index σ/μ quantifies how much observer choice matters.

In [ ]:
observers = [
    {'cx': W//2, 'cy': H//2, 'angle': 0,   'label': 'A (E-W)'},
    {'cx': W//2, 'cy': H//2, 'angle': 45,  'label': 'B (NE-SW)'},
    {'cx': W//2, 'cy': H//2, 'angle': 90,  'label': 'C (N-S)'},
    {'cx': 20,   'cy': 20,   'angle': 135, 'label': 'D (off-centre)'},
]

final_phi = history_phi[-1]
final_S   = history_entropy[-1]

fig, axes = plt.subplots(2, len(observers), figsize=(15, 6))
for col, obs in enumerate(observers):
    ts, phi_proj = observer_projection(final_phi, obs['cx'], obs['cy'], obs['angle'])
    ts, s_proj   = observer_projection(final_S,   obs['cx'], obs['cy'], obs['angle'])
    axes[0, col].plot(ts, phi_proj, color='tomato')
    axes[0, col].set_title(obs['label'], fontsize=9)
    if col == 0: axes[0, col].set_ylabel('Φ')
    axes[1, col].plot(ts, s_proj,   color='steelblue')
    if col == 0: axes[1, col].set_ylabel('S')
    axes[1, col].set_xlabel('arc length')

plt.suptitle('Observer-Dependent Projections of Φ and S', fontsize=12)
plt.tight_layout(); plt.show()

ai = anisotropy_index(final_S, observers)
print(f'Anisotropy index σ/μ = {ai:.4f}')
print('  0 → isotropic field   |  >0.1 → significant observer dependence')


## Cell 10 — Fractional Laplacian Diffusion

Scan α ∈ {0.5, 1.0, 1.5} to probe sub/super-diffusive regimes,
then run a full 300-step simulation at α = 0.65.

In [ ]:
from operators.rsvp import fractional_laplacian
from scipy.ndimage import gaussian_filter

phi_test = gaussian_filter(np.random.RandomState(0).rand(H, W), sigma=6)
alpha_scan = [0.5, 1.0, 1.5]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, af in zip(axes, alpha_scan):
    result = phi_test - 0.3 * fractional_laplacian(phi_test, alpha_frac=af)
    ax.imshow(result, cmap='magma')
    ax.set_title(f'(−∇²)^{af:.1f}')
    ax.axis('off')
plt.suptitle('Fractional Laplacian Scan', fontsize=13)
plt.tight_layout(); plt.show()

# Full evolution
phi_f, vx_f, vy_f, S_f, R_f = init_fields(H, W, seed=11)
frac_frames = []
clio_f = get_clio('standard')
for step in range(300):
    phi_f, vx_f, vy_f, S_f, R_f = evolve_fractional(
        phi_f, vx_f, vy_f, S_f, R_f,
        dt=DT, alpha_frac=0.65, clio_fn=clio_f
    )
    if step % 20 == 0:
        frac_frames.append(phi_f.copy())

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, idx, lbl in zip(axes, [0, len(frac_frames)//2, -1],
                         ['early', 'mid', 'late']):
    ax.imshow(frac_frames[idx], cmap='inferno')
    ax.set_title(f'Frac. Laplacian α=0.65 — {lbl}')
    ax.axis('off')
plt.tight_layout(); plt.show()


## Cell 11 — Soliton Kink Collision

Two tanh kinks of opposite polarity approach each other
under the double-well potential.  The collision produces
a residue pulse absorbed by the memory field R.

In [ ]:
phi_sol = gaussian_filter(np.random.RandomState(1).rand(H, W) * 0.1, sigma=4)
S_sol   = np.full((H, W), 0.05)
vx_sol  = np.zeros((H, W))
vy_sol  = np.zeros((H, W))
R_sol   = np.zeros((H, W))

phi_sol = inject_kink(phi_sol, 40, 64,  sign= 1, width=5)
phi_sol = inject_kink(phi_sol, 88, 64,  sign=-1, width=5)
vx_sol[:, :64] =  0.04   # inward
vx_sol[:, 64:] = -0.04

from operators.cosmology import double_well_force
sol_frames = []

for step in range(400):
    dw = double_well_force(phi_sol, mu=0.18)
    phi_sol, vx_sol, vy_sol, S_sol, R_sol = evolve(
        phi_sol, vx_sol, vy_sol, S_sol, R_sol,
        dt=DT, extra_phi_force=dw
    )
    if step % 20 == 0:
        sol_frames.append(phi_sol.copy())

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, idx, lbl in zip(axes, [0, len(sol_frames)//2, -1],
                         ['t=early', 't=collision', 't=late']):
    ax.imshow(sol_frames[idx], cmap='RdBu', vmin=-2, vmax=2)
    ax.set_title(f'Kink Φ — {lbl}')
    ax.axis('off')
plt.suptitle('Soliton Kink Collision', fontsize=13)
plt.tight_layout(); plt.show()


## Cell 12 — Cosmological Expansion (Hubble Drag)

Scale factor a(t) grows as de Sitter expansion.
σ(Φ) tracks structure amplitude; horizon crossing
is when a(t) crosses a target threshold.

In [ ]:
phi_c, vx_c, vy_c, S_c, R_c = init_fields(H, W, seed=99)
a_t  = 1.0
hist = CosmologicalHistory()
cos_frames = []
clio_c = get_clio('standard')

for step in range(500):
    phi_c, vx_c, vy_c, S_c, R_c, a_t = evolve_cosmological(
        phi_c, vx_c, vy_c, S_c, R_c, a_t,
        dt=DT, H0=0.003, clio_fn=clio_c
    )
    if step % 25 == 0:
        hist.record(a_t, phi_c, S_c)
        cos_frames.append(phi_c.copy())

hist.summary()

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, idx in zip(axes, [0, len(cos_frames)//2, -1]):
    ax.imshow(cos_frames[idx], cmap='plasma')
    ax.set_title(f'Φ  a={hist.scale_factors[idx]:.2f}')
    ax.axis('off')
plt.suptitle('RSVP on Expanding Background', fontsize=13)
plt.tight_layout(); plt.show()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist.scale_factors, color='gold')
ax1.set_title('Scale Factor a(t)'); ax1.set_xlabel('Frame')
ax2.plot(hist.phi_std, color='mediumorchid')
ax2.set_title('σ(Φ) — Structure Amplitude')
plt.tight_layout(); plt.show()


## Cell 13 — Coupled Dual-Manifold Synchrony

Two independent RSVP manifolds exchange a weak coupling ε.
Pearson synchrony C(t) tracks phase-locking.

In [ ]:
from operators.rsvp import init_fields as _init
phi1, vx1, vy1, S1, R1 = _init(H, W, seed=20)
phi2, vx2, vy2, S2, R2 = _init(H, W, seed=21)

sync_history = []
mm_frames    = []
clio_m = get_clio('standard')

for step in range(500):
    (phi1,vx1,vy1,S1,R1), (phi2,vx2,vy2,S2,R2), C = evolve_coupled(
        phi1,vx1,vy1,S1,R1,
        phi2,vx2,vy2,S2,R2,
        dt=DT, eps=0.04, clio_fn=clio_m
    )
    if step % 20 == 0:
        sync_history.append(C)
        mm_frames.append((phi1.copy(), phi2.copy()))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for col, idx in enumerate([0, len(mm_frames)//2, -1]):
    p1, p2 = mm_frames[idx]
    axes[0, col].imshow(p1, cmap='inferno')
    axes[0, col].set_title(f'M₁  frame {idx}'); axes[0, col].axis('off')
    axes[1, col].imshow(p2, cmap='cividis')
    axes[1, col].set_title(f'M₂  frame {idx}'); axes[1, col].axis('off')
plt.suptitle('Coupled Manifolds (ε=0.04)', fontsize=13)
plt.tight_layout(); plt.show()

plt.figure(figsize=(9, 3))
plt.plot(sync_history, color='teal')
plt.axhline(0, ls='--', color='gray')
plt.title('Inter-Manifold Synchrony C(t)')
plt.xlabel('Frame'); plt.ylabel('Pearson r')
plt.tight_layout(); plt.show()


## Cell 14 — Extractive Economic Field

Capital K self-concentrates; Labour L is depleted;
Precarity P rises with extraction.  Gini(K) is a
dynamical inequality observable.  Regulatory CLIO
damps high-gradient capital fronts.

In [ ]:
econ = EconomicField(H, W,
                     extract=0.08, accum=0.12,
                     precarity=0.06, reg=0.03,
                     seed=7)

snapshots = econ.run(steps=600, record_every=20, dt=DT)
econ.summary()

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, snap, lbl in zip(axes,
                          [snapshots[0], snapshots[len(snapshots)//2], snapshots[-1]],
                          ['early', 'mid', 'late']):
    ax.imshow(snap, cmap='YlOrRd')
    ax.set_title(f'Capital K — {lbl}')
    ax.axis('off')
plt.suptitle('Extractive Capital Field Evolution', fontsize=13)
plt.tight_layout(); plt.show()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(econ.gini_history, color='darkorange')
ax1.axhline(econ.gini_history[0], ls='--', color='gray', label='initial')
ax1.set_title('Gini Coefficient of K'); ax1.legend()

ax2.plot(econ.labour_history, color='steelblue')
ax2.set_title('Mean Labour ⟨L⟩')
plt.suptitle('Economic Metrics', fontsize=12)
plt.tight_layout(); plt.show()


## Cell 15 — Recursive Identity Constraint Surface

Add −λ(Φ − F(Φ)) to ∂ₜΦ to drive toward the fixed-point
surface of the tile-mean coarse-graining F.  The identity
gap ‖Φ − F(Φ)‖² is a consistency measure.

In [ ]:
phi_id, vx_id, vy_id, S_id, R_id = init_fields(H, W, seed=55)
LAMBDA_ID = 0.06
id_gap_history = []
id_frames = []
clio_id = get_clio('identity', lambda_id=LAMBDA_ID, tile_size=16)

for step in range(500):
    F_phi  = coarse_grain(phi_id, tile_size=16)
    gap    = phi_id - F_phi
    id_gap_history.append(float(np.mean(gap**2)))

    phi_id, vx_id, vy_id, S_id, R_id = evolve(
        phi_id, vx_id, vy_id, S_id, R_id,
        dt=DT,
        extra_phi_force=-LAMBDA_ID * gap,
        clio_fn=None   # identity pressure is the repair here
    )
    if step % 25 == 0:
        id_frames.append(phi_id.copy())

plt.figure(figsize=(9, 3))
plt.plot(id_gap_history, color='seagreen')
plt.title('Identity Gap ‖Φ − F(Φ)‖² Over Time')
plt.xlabel('Step'); plt.ylabel('Mean squared gap')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, idx, lbl in zip(axes, [0, len(id_frames)//2, -1],
                         ['early', 'mid', 'late']):
    ax.imshow(id_frames[idx], cmap='magma')
    ax.set_title(f'Identity-constrained Φ — {lbl}')
    ax.axis('off')
plt.tight_layout(); plt.show()


## Cell 16 — PCA State-Space Embedding

Project all baseline Φ snapshots to 2-D via PCA.
A compact trajectory ↔ low-dimensional attractor dynamics.

In [ ]:
coords, explained = pca_embedding(history_phi, n_components=2)

n        = len(coords)
colours  = plt.cm.plasma(np.linspace(0, 1, n))

plt.figure(figsize=(8, 7))
for i in range(n - 1):
    plt.plot(coords[i:i+2, 0], coords[i:i+2, 1],
             color=colours[i], lw=1.5)
sc = plt.scatter(coords[:, 0], coords[:, 1],
                 c=np.arange(n), cmap='plasma', s=18, zorder=5)
plt.colorbar(sc, label='Frame index')
plt.title('PCA Trajectory of Φ State Space')
plt.xlabel(f'PC1 ({100*explained[0]:.1f}% var)')
plt.ylabel(f'PC2 ({100*explained[1]:.1f}% var)')
plt.tight_layout(); plt.show()

eff_dim = 1.0 / (explained**2).sum()
print(f'Participation ratio (effective dimensionality): {eff_dim:.2f}')
print(f'PC1 explains {100*explained[0]:.2f}%  |  PC2 explains {100*explained[1]:.2f}%')


## Cell 17 — Composite Dashboard

Single-figure summary of all scalar observables tracked
during the baseline simulation.

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# Field snapshots
for col, (frames, label, cmap) in enumerate([
    (history_phi,     'Φ (scalar)',   'inferno'),
    (history_entropy, 'S (entropy)',  'viridis'),
    (history_torsion, 'ω (torsion)',  'coolwarm'),
]):
    ax = fig.add_subplot(gs[0, col])
    ax.imshow(frames[-1], cmap=cmap)
    ax.set_title(label, fontsize=10)
    ax.axis('off')

ax_adm = fig.add_subplot(gs[0, 3])
ax_adm.imshow(admissibility(history_phi[-1], history_entropy[-1]), cmap='gray')
ax_adm.set_title('Admissibility', fontsize=10)
ax_adm.axis('off')

# Metrics row
ax_e = fig.add_subplot(gs[1, :2])
ax_e.plot([m['phi_energy']   for m in metrics_log], label='Φ energy')
ax_e.plot([m['entropy_mean'] for m in metrics_log], label='S mean')
ax_e.plot([m['torsion_mean'] for m in metrics_log], label='|ω|')
ax_e.legend(fontsize=8); ax_e.set_title('Dynamical Metrics')

ax_a = fig.add_subplot(gs[1, 2:])
ax_a.plot([m['admissible_frac'] for m in metrics_log], color='seagreen')
ax_a.set_ylim(0, 1); ax_a.set_title('Admissible Fraction')

# Obstruction + structure function
obs_curve = [gluing_obstruction(f, patch=32, overlap=8)
             for f in history_phi[::5]]
ax_o = fig.add_subplot(gs[2, :2])
ax_o.plot(obs_curve, color='crimson')
ax_o.set_title('Sheaf Obstruction Ω(t)')

lags, D = structure_function(history_phi[-1], max_lag=32)
ax_sf = fig.add_subplot(gs[2, 2:])
ax_sf.loglog(lags, D, marker='o', ms=3, color='royalblue')
ax_sf.set_title('Structure Function D(r)')
ax_sf.set_xlabel('Lag r')

plt.suptitle('RSVP Field Laboratory — Composite Dashboard', fontsize=14, y=1.01)
plt.show()
